# Appendix: TRF Repeat Panel Construction

This appendix documents how the repeat locus panel used in Notebook 7 was built.
[Tandem Repeat Finder](https://tandem.bu.edu/trf/trf.html) (TRF, Benson 1999) is a
published third-party tool that we install and orchestrate but don't modify.       
This notebook walks through installing TRF, parsing and filtering its raw output, and the CLI commands to reproduce the curated panel end-to-end.   

Parsing and filtering functions are implemented in `nwflex.trf`.

In [1]:
# 🧙 Notebook magic: autoreload modules
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

from nwflex.trf import (
    parse_trf_dat,
    annotate_repeat_isolation,
    filter_isolated_repeats
)

DATA_DIR = Path("../data")

## Running TRF

We use [Tandem Repeat Finder](https://tandem.bu.edu/trf/trf.html) (TRF) to identify
repetitive regions in the reference genome.      
TRF scans a FASTA for tandem repeats and
produces a `.dat` file with one row per detected repeat tract: coordinates, period
size, copy number, percent matches, and the consensus motif sequence.

### Installation

```bash
conda install -c bioconda trf
```

**samtools** is also needed to index the reference FASTA
(only needed if running from your own genome; skip if using the pre-built panel):

```bash
conda install -c bioconda samtools
```

### TRF command

The command below runs TRF on a single chromosome FASTA. The numeric parameters
control the match/mismatch weights and the minimum alignment score — these are the
standard settings used throughout this paper:

```bash
trf <chrom.fa> 2 5 5 80 10 20 100 -f -d -m -ngs > <chrom.trf.dat>
#               ^ match weight
#                 ^ mismatch penalty
#                   ^ indel penalty
#                     ^^ match probability (for scoring)
#                        ^^ indel probability
#                           ^^ minimum alignment score
#                              ^^^ maximum period size (bp)
```

## Parsing and filtering

The raw TRF output includes nested, imperfect, and closely-spaced repeats. For our
simulations, we keep only **isolated, perfect** loci:

- **isolated** — at least 100 bp of uninterrupted flank on each side, and no
  overlapping neighbor tracts.
- **perfect** — 100% matches to the consensus motif (no internal mismatches or
  indels). 

Three functions from `nwflex.trf` handle this:

- `parse_trf_dat(path)` — reads a `.dat` file into a DataFrame, one row per detected repeat
- `annotate_repeat_isolation(df, chrom_lengths)` — adds `ldist` / `rdist` (distance to nearest neighbor) and per-locus overlap count
- `filter_isolated_repeats(df, ...)` — keeps only loci that pass isolation, purity, and period-size criteria

In [2]:
# Demo: parse the included fixture (a chr21 snippet)
fixture = DATA_DIR / "chr21_snippet_demo.dat"
raw = parse_trf_dat(fixture)
print(f"Raw TRF rows: {len(raw)}")

# `annotate_repeat_isolation` needs each chromosome's length to compute the
# right-flank distance (distance from the repeat's end to the end of the
# chromosome). In the real pipeline, `make_trf_panel.py` reads these from the
# FASTA's `.fai` index via `read_chrom_lengths(fasta_path)`. Here we hardcode
# just chr21's hg38 length since the fixture only spans chr21.
chrom_lengths = {"chr21": 46_709_983}
annotated = annotate_repeat_isolation(raw, chrom_lengths)

# Match the pre-built panel: isolated (min_dist=100), mono/di/tri only
# (max_period=3), and perfect mono-runs (perfect_only=True). The pre-built
# panel additionally drops imperfect multi-base periods via the `iso_pure`
# class in `make_trf_panel.py`; here we approximate that with pct_matches==100.
filtered = filter_isolated_repeats(
    annotated, min_dist=100, max_period=3, perfect_only=True
)
filtered = filtered[filtered["pct_matches"] == 100].copy()
print(f"After isolation + max_period=3 + purity filters: {len(filtered)} loci")

filtered[["chrom", "start", "end", "period_size", "copy_number",
          "consensus_pattern", "ldist", "rdist"]]

Raw TRF rows: 200
After isolation + max_period=3 + purity filters: 4 loci


,chrom,start,end,period_size,copy_number,consensus_pattern,ldist,rdist
8,chr21,5011757,5011767,3,3.3,CTG,107,147
92,chr21,5024617,5024637,1,20.0,T,932,223
118,chr21,5030912,5030922,1,10.0,A,131,157
168,chr21,5039929,5039949,1,20.0,A,149,149


## CLI to reproduce the pre-built panel

The pre-built panel at `data/hg38_motif_sample_K100.tsv` contains 6,900 loci, with up to
100 loci per mono-, di-, and trinucleotide motif from hg38. All loci in the panel pass the isolation and purity criteria defined above. The
`lflank` and `rflank` columns hold the raw genomic flank sequence; Notebook 7 takes the
innermost `FLANK_LEN` bp from each side for simulation.

The full TRF output the panel was built from is several hundred MB — too large to ship
in the repository. That's why the parsing demo above uses a 200-locus snippet from
chr21. The three stages below reproduce the panel end-to-end, starting from a hg38
FASTA.

The `--fasta` argument in stage 2 expects hg38 with chrY non-PAR regions hard-masked,
so that only the PAR (which behaves autosomally because it recombines with chrX)
contributes chrY loci. To skip the masking step, drop `chrY` from `--chr` and pass a
standard `hg38.fa`.

**Stage 1.** Run TRF on each chromosome's FASTA (see *Running TRF* above for the
parameter meanings). Assuming one per-chromosome FASTA per file (`hg38.chr1.fa`,
`hg38.chr2.fa`, ...):

```bash
for chr in chr1 chr2 chr4 chr6 chr8 chr10 chr12 chr13 chr14 chr15 \
           chr16 chr19 chr20 chr21 chr22 chrY; do
    trf hg38.${chr}.fa 2 5 5 80 10 20 100 -f -d -m -ngs > ${chr}.trf.dat
done
```

Collect the resulting `.dat` files into a single directory; that path is passed to
`--trf-dir` in stage 2.

**Stage 2.** Build a large iso-pure panel from the TRF calls — a reusable intermediate
covering all mono- through hexanucleotide loci that pass the isolation and purity
filters. The intermediate path is yours to choose; it only feeds stage 3.

```bash
python scripts/panels/make_trf_panel.py \
    --chr chr1 chr2 chr4 chr6 chr8 chr10 chr12 chr13 chr14 chr15 \
          chr16 chr19 chr20 chr21 chr22 chrY \
    --fasta <path/to/hg38.fa> \
    --trf-dir <path/to/trf_dat_dir> \
    --require-pure \
    --max-period 6 \
    --max-tract-bp 1000 \
    --output iso_pure_panel.tsv
```

**Stage 3.** Restrict to mono/di/trinucleotide motifs and take the first 100
boundary-clean loci per motif.

```bash
python scripts/panels/build_motif_catalog.py \
    --panel-tsv iso_pure_panel.tsv \
    --K 100 \
    --max-period 3 \
    --output data/hg38_motif_sample_K100.tsv
```

In principle the catalog could contain 76 motifs × 100 = 7,600 loci (4 mono + 12 di +
60 tri, after excluding rotations that collapse to a shorter period). In practice some
motifs — particularly CG-containing ones — have fewer than 100 isolated, perfect loci
in hg38, so the final catalog contains 6,900 loci.

In [3]:
PANEL_TSV = DATA_DIR / "hg38_motif_sample_K100.tsv"
if not PANEL_TSV.exists():
    raise FileNotFoundError(
        f"Panel not found at {PANEL_TSV}\n"
        "Run this notebook from the notebooks/ directory, or update DATA_DIR in the setup cell."
    )

panel = pd.read_csv(PANEL_TSV, sep="\t")
print(f"{len(panel)} loci in pre-built panel")
print("Columns:", list(panel.columns))
panel.head(10)

6900 loci in pre-built panel
Columns: ['pind', 'chr', 'start_38', 'stop_38', 'strand', 'type', 'lflank', 'rflank', 'ms_seq', 'ref_score_per_base']


,pind,chr,start_38,stop_38,strand,type,lflank,rflank,ms_seq,ref_score_per_base
0,0,chr1,36351,36364,+,A,CCATCTCTGGGCCCAGAATGACCCACTGGAGACCTTACAGCTCTCC...,CCCAGCCTGGCGGAAAGAATTTAAATTATAAAAACTTAGAAGTATG...,AAAAAAAAAAAAA,1.0
1,1,chr1,51864,51877,+,A,GTGGGGGTTGAGTTTCACTTTATTTAAAGTGAGTCTTAATCCTCCA...,GAAGATTGATCAGAGAGTACCTCCCCTAAGGGTACATGCAGATAAA...,AAAAAAAAAAAAA,1.0
2,2,chr1,71175,71186,+,A,AGTATATTACTTGGATCCATCTATGTCATTTTCCATGGTTAATGTT...,CCTTAACAAATGATTCTGACAAATATCTTCTCTTTCCAGGGAGAAT...,AAAAAAAAAAA,1.0
3,3,chr1,77174,77195,+,A,CTTCATGTCTAAAACACCGAGAGAGGCACTCTTATGCATTGTTGGT...,GGAAAATAACCAAATGACAATTAGTGAGTACTACTTGCAAAACTTG...,AAAAAAAAAAAAAAAAAAAAA,1.0
4,4,chr1,108545,108561,+,A,TGTACCATGCTCCTCCTTAATCATTCTGAGGTTACATCTTAAGTCC...,GAATGGAGAGAATGCTACATGAGAGAAAGGATCTTATCTATCATGT...,AAAAAAAAAAAAAAAA,1.0
5,5,chr1,144887,144899,+,A,CCTGTAATCCCAGCTACTCGGGAGGCTGAGGCAGGAGAATTGCTTG...,GGGTATTAATTTTTACAGAGGATCAGCACAATGAGGGACACACTAG...,AAAAAAAAAAAA,1.0
6,6,chr1,147591,147601,+,A,AATTTTTAAATATTCTGTAGAGACAAGGTCTTGCTAGGTTGCCCAG...,CAGATAATGGCAAATGTTGGTGAAGGCCGGGCATGGTGGCAGCCTG...,AAAAAAAAAA,1.0
7,7,chr1,157107,157121,+,A,AGTGATGTTCAATCACCATGTACGTATCTTGAAGGATATGGCCCAT...,GGACAATAAAGAAATAAAGCTAATAAGCTAACATAAGGAAAGATAA...,AAAAAAAAAAAAAA,1.0
8,8,chr1,164673,164695,+,A,CTTAATTAAATACAACCCTAGTGGTGAATGACTAAAGATGGATTAC...,TTCCTTTGGGAAGGCCTTCTACATAAAAATCTTCAACATGAGACTG...,AAAAAAAAAAAAAAAAAAAAAA,1.0
9,9,chr1,165287,165310,+,A,CGAGCTATAAGAAAAAAAAGAAAAAGGGATATCATTTAAACACAGT...,CAGCTAGCAGGTGACATTTGCTATAGGGAGACTAGGGATATGATCT...,AAAAAAAAAAAAAAAAAAAAAAA,1.0


In [4]:
# Motif counts broken down by period length
counts = panel.groupby("type").size()
by_period = counts.groupby(counts.index.str.len()).agg(["count", "sum"])
by_period.columns = ["n_motifs", "n_loci"]
by_period.index.name = "period_bp"

print(f"{len(counts)} distinct motifs, {counts.sum()} total loci")
print(f"Motifs with < 100 loci: {(counts < 100).sum()}")
by_period

76 distinct motifs, 6900 total loci
Motifs with < 100 loci: 10


,n_motifs,n_loci
period_bp,,
1,4,400
2,12,1080
3,60,5420


## Summary

This appendix produced **`data/hg38_motif_sample_K100.tsv`** — a curated panel of:

- **6,900 loci** across **76 motifs** (mono/di/trinucleotide)
- **Isolated**: ≥ 100 bp uninterrupted flank on each side, no overlapping neighbors
- **Perfect**: 100% match to the consensus motif
- **500 bp** of genomic flank stored on each side (`lflank` / `rflank`)

Notebook 7 consumes this panel directly. The parsing and filtering functions in `nwflex.trf` are also usable on any TRF `.dat` output if you want to build a custom panel with different period, purity, or isolation thresholds.
